# **Unigrams: Full Workflow**
The primary purpose of the unigram workflow is to generate a vocabulary **whitelist** that can be used to filter out unwanted words from a multigram corpus. The workflow consists of two steps: (1) downloading the unigram corpus into a database and (2) filtering and normalizing the corpus and generating the whitelist.

## **Setup**
### Imports

In [1]:
%load_ext autoreload
%autoreload 2

from ngramprep.ngram_filter import FilterConfig, PipelineConfig, load_stopwords
from ngramprep.ngram_filter.lemmatizer import CachedSpacyLemmatizer
from ngramprep.ngram_acquire import download_and_ingest_to_rocksdb
from ngramprep.ngram_filter.pipeline.orchestrator import build_processed_db
from ngramprep.utilities.peek import db_head, db_peek, db_peek_prefix

### Configure
Here we set basic parameters: the corpus to download, the size of the ngrams to download, and the size of the year bins.

In [2]:
db_path_stub = '/scratch/edk202/NLP_corpora/Google_Books/'
archive_path_stub = None
release = '20200217'
language = 'eng'
ngram_size = 1
bin_size = 1

## **Step 1: Download and Ingest**

In [3]:
download_and_ingest_to_rocksdb(
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    archive_path_stub=archive_path_stub,
    ngram_type="tagged",
    overwrite_db=True,
    open_type="write:packed24",
    compact_after_ingest=True
)

N-GRAM ACQUISITION PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-06 20:24:48

Download Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Ngram repo:           https://books.storage.googleapis.com/?prefix=ngrams/books/20200217/eng/1-
DB path:              /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams.db
File range:           0 to 23
Total files:          24
Files to get:         24
Skipping:             0
Download workers:     24
Batch size:           50,000
Ngram size:           1
Ngram type:           tagged
Overwrite DB:         True
DB Profile:           write:packed24

Download Progress
════════════════════════════════════════════════════════════════════════════════════════════════════


Files Processed: 100%|█████████████████████████████████████████████████████████| 24/24 [05:03<00:00]


Post-Ingestion Compaction
════════════════════════════════════════════════════════════════════════════════════════════════════
Initial DB size:         45.87 GB


Compaction completed in 0:03:16
Size before:             45.87 GB
Size after:              57.76 GB
Space saved:             -11.89 GB (-25.9%)

Processing complete!

Final Summary
════════════════════════════════════════════════════════════════════════════════════════════════════
Fully processed files:       24
Failed files:                0
Total entries written:       41,783,218
Write batches flushed:       24
Uncompressed data processed: 43.28 GB
Processing throughput:       88.46 MB/sec

End Time: 2026-02-06 20:33:09.017865
Total Runtime: 0:08:21.014752
Time per file: 0:00:20.875615
Files per hour: 172.5


 ## **Step 2: Filter, Normalize, and Generate Whitelist**
`config.py` contains generic defaults for the filtering pipeline. You can override these defaults by passing option dictionaries to the `build_processed_db` function, as seen below. By default, we:
1. case-normalize the tokens
2. remove tokens containing non-alphanumeric text
3. remove stopwords using the `stop-words` package
4. lemmatize the tokens using the `spaCy` package—first using a lookup table and falling back to rules when lookups fail
5. Create a whitelist of the top 20,000 most frequent words that pass a `pyenchant` spell-check and appear in all corpora from 1900–2019 (inclusive).

In [4]:
stop_set, stop_lang = load_stopwords("ru")

filter_config = FilterConfig(
    stop_set=stop_set,
    stop_words_language=stop_lang,
    lemma_gen=CachedSpacyLemmatizer(language="ru"),
    ascii_alpha_only=True,
    min_context_tokens=1,
    min_len=3,
    bin_size=bin_size
)

pipeline_config = PipelineConfig(
    # Path construction
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    # Pipeline options
    mode="restart",
    num_workers=20,
    num_initial_work_units=300,
    cache_partitions=True,
    use_cached_partitions=False,
    progress_every_s=5,
    compact_after_ingest=True,
    # Output whitelist options
    output_whitelist_path="default",
    output_whitelist_top_n=30_000,
    output_whitelist_year_range=(1900, 2019),
    output_whitelist_spell_check=True,
    output_whitelist_spell_check_language="ru_RU"
)

build_processed_db(
    filter_config=filter_config,
    pipeline_config=pipeline_config
);


N-GRAM FILTER PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-06 20:33:52
Mode:       RESTART

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Source DB:            /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams.db
Target DB:            ...dk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/1grams_processed.db
Temp directory:       ...tch/edk202/NLP_corpora/Google_Books/20200217/eng/1gram_files/processing_tmp

Parallelism
────────────────────────────────────────────────────────────────────────────────────────────────────
Workers:              20
Initial work units:   300

Database Profiles
────────────────────────────────────────────────────────────────────────────────────────────────────
Reader profile:       read:packed24
Writer profile:       write:packed24

Ingestion Configuration
────────────────────────

Shards Ingested: 100%|███████████████████████████████████████████████████████| 300/300 [06:50<00:00]



Ingestion complete: 300 shards, 18,386,458 items in 410.2s (44,818 items/s)

Phase 4: Finalizing database...
════════════════════════════════════════════════════════════════════════════════════════════════════

Post-Ingestion Compaction
────────────────────────────────────────────────────────────────────────────────────────────────────
Initial DB size:         26.53 GB
Compaction completed in 0:00:51
Size before:             26.53 GB
Size after:              22.09 GB
Space saved:             4.44 GB (16.7%)

Phase 5: Generating output whitelist...
════════════════════════════════════════════════════════════════════════════════════════════════════
  Output path: ...LP_corpora/Google_Books/20200217/eng/1gram_files/1grams_processed.db/whitelist.txt
  Extracting top 30,000 tokens
  Spell checking enabled (ru_RU)
  Year range filter: 1900-2019 (inclusive)
  Generated whitelist with 0 tokens in 246.1s

┌────────────────────────────────────────────────────────────────────────────────────────

## **Optional: Inspect Database Files**

### `db_head`: Show first N records

In [5]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_head(db, n=5)

First 5 key-value pairs:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   aaa
     Value: Total: 4,449,225 occurrences in 1,052,377 volumes (1477-2019, 402 bins)

[ 2] Key:   aaaa
     Value: Total: 472,912 occurrences in 91,764 volumes (1477-2019, 337 bins)

[ 3] Key:   aaaaa
     Value: Total: 54,371 occurrences in 22,966 volumes (1581-2019, 274 bins)

[ 4] Key:   aaaaaa
     Value: Total: 14,571 occurrences in 10,476 volumes (1608-2019, 237 bins)

[ 5] Key:   aaaaaaa
     Value: Total: 6,883 occurrences in 5,204 volumes (1653-2019, 197 bins)



### `db_peek`: Show records starting from a key

In [6]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek(db, start_key="police", n=5)

5 key-value pairs starting from 706f6c696365:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   police
     Value: Total: 160,399,233 occurrences in 11,467,994 volumes (1478-2019, 398 bins)

[ 2] Key:   policea
     Value: Total: 403 occurrences in 309 volumes (1804-2019, 125 bins)

[ 3] Key:   policeaan
     Value: Total: 139 occurrences in 63 volumes (1860-1996, 32 bins)

[ 4] Key:   policeability
     Value: Total: 269 occurrences in 125 volumes (1963-2019, 24 bins)

[ 5] Key:   policeable
     Value: Total: 1,433 occurrences in 1,147 volumes (1865-2019, 77 bins)



### `db_peek_prefix`: Show records matching a prefix

In [7]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek_prefix(db, prefix="doctor", n=5)

5 key-value pairs with prefix 646f63746f72:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   doctor
     Value: Total: 92,297,746 occurrences in 10,803,973 volumes (1476-2019, 482 bins)

[ 2] Key:   doctora
     Value: Total: 22,165 occurrences in 11,203 volumes (1644-2019, 215 bins)

[ 3] Key:   doctoraal
     Value: Total: 5,332 occurrences in 2,015 volumes (1878-2019, 94 bins)

[ 4] Key:   doctoraalexamen
     Value: Total: 441 occurrences in 190 volumes (1919-2018, 56 bins)

[ 5] Key:   doctoraalscriptie
     Value: Total: 934 occurrences in 674 volumes (1961-2019, 53 bins)

